# 01 - Silver: ED Performance

**Pipeline:** Bronze -> Silver  
**Source:** AIHW MyHospitals API - measure MYH0005 (4-hour departure rate), MYH0010, MYH0011, MYH0013  
**Target:** `silver.fact_ed_performance` (Delta table)  

Steps:
1. Read raw JSON landed by ADF from bronze layer
2. Profile schema
3. Filter to WA hospitals
4. Cast types and drop nulls
5. Write as Delta table

In [ ]:
# ------------------------------------------------------------------
# 1. Read raw JSON from bronze layer and flatten nested schema
# ------------------------------------------------------------------
MEASURE_CODES = ["MYH0005", "MYH0010", "MYH0011", "MYH0013"]

# Absolute OneLake paths - works without lakehouse attached to notebook
WORKSPACE_ID = "e53f915a-de32-40a9-9b16-af4486796bbe"
LAKEHOUSE_ID = "6383e12e-91b9-4df2-99c5-06c9597bc27e"
ONELAKE_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
SILVER_PATH  = f"{ONELAKE_ROOT}/Tables/silver/fact_ed_performance"

from pyspark.sql.functions import col, lit, explode, size
from functools import reduce
from pyspark.sql import DataFrame

dfs = []
for code in MEASURE_CODES:
    path = f"{ONELAKE_ROOT}/Files/bronze/aihw/measures/{code}/raw.json"
    raw = spark.read.option("multiline", "true").json(path)

    # Explode result array and flatten reporting_unit_summary nested struct
    df = raw.select(explode(col("result")).alias("r")).select(
        col("r.data_set_id"),
        col("r.value"),
        col("r.lower_value"),
        col("r.upper_value"),
        col("r.suppressions"),
        col("r.reporting_unit_summary.reporting_unit_code").alias("hospital_code"),
        col("r.reporting_unit_summary.reporting_unit_name").alias("hospital_name"),
        col("r.reporting_unit_summary.reporting_unit_type.reporting_unit_type_code").alias("unit_type")
    ).withColumn("measure_code", lit(code))

    row_count = df.count()
    dfs.append(df)
    print(f"{code}: {row_count:,} rows")

combined = reduce(DataFrame.unionByName, dfs)
print(f"\nTotal rows across all measures: {combined.count():,}")

## Step 1 - Load Bronze Data

**What:** Reads raw JSON files from OneLake bronze layer, uploaded by `scripts/ingest_bronze.py`.

**Data shape:** Each record in the API response has a nested `reporting_unit_summary` struct. We flatten it here.

**Reproducibility:** If data is stale, re-run `python3 scripts/ingest_bronze.py` to refresh bronze.

## Step 2 - Profile Schema

**What:** Print the inferred schema and sample rows from the raw bronze data.

**Why:** Confirms the data landed correctly from the API and that no structural changes have occurred between API versions. Review this output before every ingestion cycle.

## Step 3 - Filter to WA + Join Time Periods

**What:** Two joins to add context not in the data-items endpoint:
1. Join with `wa_hospitals.json` (hospital codes) to keep only WA hospitals
2. Join with `datasets.json` (time period lookup) on `data_set_id` to add `reporting_start_date` / `reporting_end_date`

**Design decision:** AIHW `data-items` does not include state or time period directly. Both come from separate bronze files ingested by `ingest_bronze.py`. No API calls in the transformation layer.

**Suppression:** The API uses a `suppressions` array (not a boolean). A record is suppressed if `size(suppressions) > 0`. Suppressed records are excluded from silver.

## Step 4 - Data Quality Checks

**What:** Validate the silver DataFrame before writing.

**Checks run:**
- `MYH0005` values must be within `[0, 100]` (percentage)
- Hospital and measure counts must be non-zero
- Date range is printed to confirm data freshness

**Expected results:** 0 invalid percentage values. If checks fail, investigate the raw bronze data or re-run ingestion.

## Step 5 - Write Silver Delta Table

**What:** Writes the cleaned DataFrame as a Delta table in OneLake and registers it in the lakehouse metastore.

**Mode:** `overwrite` - safe to re-run. Each run produces a full refresh from the latest bronze data.

**Output table:** `silver.fact_ed_performance`

**Next step:** Run `02_silver_dim_hospitals.ipynb`, then `03_gold_ed_trends.ipynb`.

In [ ]:
# ------------------------------------------------------------------
# 2. Profile schema - understand the raw structure
# ------------------------------------------------------------------
combined.printSchema()
combined.show(5, truncate=False)

In [ ]:
# ------------------------------------------------------------------
# 3. Filter to WA hospitals and join time periods from datasets
# ------------------------------------------------------------------
from pyspark.sql.functions import to_date

# Load WA hospital codes (pre-filtered by ingest script)
wa_raw = spark.read.option("multiline", "true").json(
    f"{ONELAKE_ROOT}/Files/bronze/aihw/reporting_units/wa_hospitals.json"
)
wa_codes = (
    wa_raw.select(explode(col("result")).alias("h"))
    .select(col("h.reporting_unit_code").alias("hospital_code"))
)
print(f"WA hospital codes loaded: {wa_codes.count()}")

# Load datasets lookup (data_set_id -> time period)
ds_raw = spark.read.option("multiline", "true").json(
    f"{ONELAKE_ROOT}/Files/bronze/aihw/datasets/datasets.json"
)
datasets = (
    ds_raw.select(explode(col("result")).alias("d"))
    .select(
        col("d.data_set_id"),
        to_date(col("d.reporting_start_date")).alias("time_period_start"),
        to_date(col("d.reporting_end_date")).alias("time_period_end")
    )
)
print(f"Dataset records loaded: {datasets.count()}")

# Filter combined to hospital-type WA records only
hospitals_only = combined.filter(col("unit_type") == "H")
wa_only = hospitals_only.join(wa_codes, "hospital_code", "inner")
print(f"WA hospital rows (before time join): {wa_only.count()}")

# Join time periods
silver_df = (
    wa_only
    .join(datasets, "data_set_id", "left")
    .select(
        col("hospital_code"),
        col("hospital_name"),
        col("measure_code"),
        col("time_period_start"),
        col("time_period_end"),
        col("value").cast("double"),
        col("lower_value").cast("double"),
        col("upper_value").cast("double"),
        # suppressed = suppressions array is non-empty
        (size(col("suppressions")) > 0).alias("suppressed")
    )
    .filter(col("value").isNotNull())
    .filter(col("time_period_start").isNotNull())
    # Exclude suppressed values
    .filter(col("suppressed") == False)
)

print(f"\nFinal WA silver rows: {silver_df.count():,}")
silver_df.show(10)

In [ ]:
# ------------------------------------------------------------------
# 4. Data quality checks before writing
# ------------------------------------------------------------------
pct = silver_df.filter(col("measure_code") == "MYH0005")
invalid = pct.filter((col("value") < 0) | (col("value") > 100)).count()
print(f"MYH0005 invalid percentage values (outside 0-100): {invalid}")

print(f"Distinct WA hospitals: {silver_df.select('hospital_code').distinct().count()}")
print(f"Distinct measures:     {silver_df.select('measure_code').distinct().count()}")

dates = silver_df.agg(
    {"time_period_start": "min", "time_period_start": "max"}
).collect()[0]
print(f"Date range: {silver_df.agg({'time_period_start': 'min'}).collect()[0][0]} "
      f"to {silver_df.agg({'time_period_start': 'max'}).collect()[0][0]}")

In [ ]:
# ------------------------------------------------------------------
# 5. Write silver Delta table
# ------------------------------------------------------------------
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_PATH)

# Register as table in the lakehouse metastore
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS silver.fact_ed_performance
    USING DELTA
    LOCATION '{SILVER_PATH}'
""")

print("silver.fact_ed_performance written successfully")
print(f"Location: {SILVER_PATH}")
print(f"Final row count: {spark.table('silver.fact_ed_performance').count()}")